In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

GATEWAY_URL = os.getenv('GATEWAY_URL')
MEMORY_ID = os.getenv('MEMORY_ID')

REGION = os.getenv("AWS_REGION")

GATEWAY_URL = os.getenv("GATEWAY_URL")

CLIENT_ID = os.getenv("COGNITO_CLIENT_ID")

CLIENT_SECRET = os.getenv("COGNITO_CLIENT_SECRET")
    
USERNAME = os.getenv("COGNITO_USERNAME")

PASSWORD = os.getenv("COGNITO_PASSWORD")

USER_POOL_ID = os.getenv("COGNITO_USER_POOL_ID")

### Prerequisite IAM roles
* Reesources to create within AWS account
    - AWS Lambda function 
    - AWS Lambda Execution IAM Role
    - AgentCore Gateway IAM Role
    - Cognito User Pool and User Pool Client

In [2]:
# Import libraries
import boto3

from mcp.client.streamable_http import streamablehttp_client

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


### Get Auth token and Create MCP Client

### Get Auth token

In [3]:
import boto3
import base64
import hashlib
import hmac

# COGNITO CLIENT
cognito_client = boto3.client(
    "cognito-idp",
    region_name=REGION
)

# SECRET HASH
message = bytes(USERNAME + CLIENT_ID, "utf-8")

key = bytes(CLIENT_SECRET, "utf-8")
secret_hash = base64.b64encode(
    hmac.new(
        key,
        message,
        digestmod=hashlib.sha256
    ).digest()
).decode()

import boto3

# Set the user as admin in order to get the auth token and skip password change for the user in cognito
cognito_client = boto3.client(
    "cognito-idp",
    region_name=REGION
)

response = cognito_client.admin_set_user_password(
    UserPoolId=USER_POOL_ID,
    Username=USERNAME,
    Password=PASSWORD,
    Permanent=True
)

print("✅ Permanent password set successfully")

✅ Permanent password set successfully


In [4]:
# AUTHENTICATE
auth_response = cognito_client.initiate_auth(
    ClientId=CLIENT_ID,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={
        "USERNAME": USERNAME,
        "PASSWORD": PASSWORD,
        "SECRET_HASH": secret_hash,
    },
)

# ACCESS TOKEN
ACCESS_TOKEN = (
    auth_response["AuthenticationResult"]["AccessToken"]
)

print("✅ Access Token Generated")

✅ Access Token Generated


### Create MCP Client

In [5]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {ACCESS_TOKEN}"
        },
    )
)

print("✅ Secure MCP Client Created")

✅ Secure MCP Client Created


### Create Memory Session Manager

In [6]:
import uuid

from strands import Agent
from strands.models import BedrockModel

from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager
)

from strands_memory import (
    create_memory_config,
)

from strands_agents import (
    SYSTEM_PROMPT,
    MODEL_ID,
)

ACTOR_ID = os.getenv('ACTOR_ID')
SESSION_ID = f"{ACTOR_ID}-{uuid.uuid4()}"

model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION
)

# MEMORY CONFIG
memory_config = create_memory_config(
    memory_id=MEMORY_ID,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID
)

# SESSION MANAGER
session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=memory_config,
    region_name=REGION
)

print("✅ Memory Configured")

✅ Product comparison tool ready
✅ Memory Configured


### Create Agent

In [7]:
# CREATE MCP TOOLS
with mcp_client:

    mcp_tools = mcp_client.list_tools_sync()

    print("\n=== MCP TOOLS DISCOVERED ===")

    for tool in mcp_tools:
        print(tool.tool_name)

# CREATE AGENT ONLY ONCE
agent = Agent(
    model=model,
    tools=mcp_tools,
    session_manager=session_manager,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent Created Successfully")


=== MCP TOOLS DISCOVERED ===
target-quick-start-dec3b8___compare_products
target-quick-start-dec3b8___get_product_details
target-quick-start-dec3b8___search_products
✅ Agent Created Successfully


### Test Agent

In [10]:
test_prompts = [

    "List all your tools",

    "Suggest Lenovo gaming laptops",

    "Tell me detailed information about HP laptops",

    "Compare Lenovo Legion 5 and Dell XPS 13",

    "I prefer laptops under 1 lakh budget",

    "What brand preference did I mention earlier?"
]

def test_agent_responses(prompts):

    for i, prompt in enumerate(prompts, 1):

        print(f"\n==============================")
        print(f"Test Case {i}")
        print(f"==============================")

        print(f"\nPrompt:\n{prompt}")

        print("\nResponse:")
        print("-" * 50)

        try:
            response = agent(prompt)
            print(response)
        except Exception as e:
            print(f"Error: {str(e)}")

        print("-" * 50)

# RUN TESTS

test_agent_responses(test_prompts)


Test Case 1

Prompt:
List all your tools

Response:
--------------------------------------------------


Here are the tools I have available:

1. **search_products**
   - **Description**: Search products based on category, optional budget, and optional brand preference.
   - **Parameters**:
     - **category** (required): Product category to search for.
     - **brand** (optional): Optional preferred brand name.
     - **budget** (optional): Optional budget range for products.

2. **get_product_details**
   - **Description**: Retrieve detailed specifications and features of a product.
   - **Parameters**:
     - **product_name** (required): Name of the product.

3. **compare_products**
   - **Description**: Compare two products and provide recommendation summary.
   - **Parameters**:
     - **product1** (required): First product name.
     - **product2** (required): Second product name.

Let me know if you need help with something specific!

Here are the tools I have available:

1. **